# run_baseline_comparison_gate

该 Notebook 是 `baseline_comparison_gate` 的 Colab 执行入口。它只负责 Colab 会话编排, 正式逻辑全部委托给仓库脚本和 `experiments/baseline_comparison_gate/` 模块。

当前 Notebook 的目标是完成阶段三正式 runner 前置准备: 检查并解压阶段二 real-video VAE 正式结果包、拉取外部 baseline 上游源码、执行真实 smoke、汇总 real smoke、冻结 formal input contract, 并生成正式 scoring work-item 计划。当前 work-item 计划不包含外部 baseline 分数, 不能支持论文 claim。


## 0. 使用边界

- 本 Notebook 不直接写出正式 `records`、`thresholds`、`tables`、`figures` 或 `reports` 逻辑。
- 本 Notebook 不把 `external_baselines/` 提交进仓库。
- `external_baselines/` 是冷启动缓存目录, 每次 Colab 运行时通过 `scripts/prepare_baselines/fetch_external_baselines.py` 重新拉取。
- 真实 baseline adapter、权重 digest 和正式比较表仍需要后续阶段继续实现。

In [ ]:
# 可修改配置区
REPO_URL = "https://github.com/RICHAAARC/TSTW-VW.git"
REPO_DIR = "/content/TSTW-VW"
GIT_REF = "explicit-sync"
DRIVE_MOUNT = "/content/drive"
DRIVE_RESULT_ROOT = "/content/drive/MyDrive/TSTW/results"
BASELINE_CONFIG_DIR = f"{REPO_DIR}/configs/baselines"
EXTERNAL_BASELINES_ROOT = f"{REPO_DIR}/external_baselines"
SESSION_RUN_ROOT = "/content/TSTW_runtime/runs/baseline_comparison_smoke"
REAL_VIDEO_VAE_PACKAGE_ROOT = "/content/drive/MyDrive/TSTW/results/real_video_vae_latent_probe_stage2_final_formal_audit/real_video_vae_latent_probe_formal_20260611T012845Z_2dbc783"
LOCAL_REAL_VIDEO_VAE_EXTRACT_ROOT = "/content/TSTW_runtime/input_packages"
REAL_VIDEO_VAE_INPUT_CHECK_PATH = "/content/TSTW_runtime/input_checks/real_video_vae_for_baseline_check.json"
EXTRACT_REAL_VIDEO_VAE_PACKAGE = True  # 是否检查并解压阶段二结果包。
RUN_VIDEOSEAL_REAL_SMOKE = True  # 是否运行 external_videoseal 的真实 smoke。
VIDEOSEAL_INPUT_VIDEO_PATH = ""  # external_videoseal smoke 的可选输入视频路径。留空表示脚本自动生成确定性测试视频; 填入路径则使用该真实视频做 smoke。
RUN_RIVAGAN_REAL_SMOKE = True  # 是否运行 external_rivagan 的真实 smoke。
RIVAGAN_INPUT_VIDEO_PATH = ""  # external_rivagan smoke 的可选输入视频路径。留空表示脚本自动生成确定性测试视频; 填入路径则使用该真实视频做 smoke。
RUN_HIDDEN_FRAMEWISE_REAL_SMOKE = True  # 是否运行 external_hidden_framewise 的真实 smoke。
HIDDEN_FRAMEWISE_INPUT_VIDEO_PATH = ""  # external_hidden_framewise smoke 的可选输入视频路径。留空表示脚本自动生成确定性测试视频; 填入路径则使用该真实视频做 smoke。
RUN_BASELINE_REAL_SMOKE_SUMMARY = True  # 是否汇总三个 real smoke 结果。
BASELINE_REAL_SMOKE_SUMMARY_DIR = "/content/drive/MyDrive/TSTW/results/baseline_comparison_gate/baseline_real_smoke_summary_latest"
BASELINE_FORMAL_RUN_ROOT = "/content/TSTW_runtime/runs/baseline_comparison_formal"
EXTRACTED_REAL_VIDEO_VAE_PACKAGE_ROOT = f"{LOCAL_REAL_VIDEO_VAE_EXTRACT_ROOT}/real_video_vae_latent_probe_formal"
RUN_BASELINE_FORMAL_SCORING_PLAN = True  # 是否生成正式 scoring work-item 计划。该步骤只生成任务清单, 不执行外部 baseline 检测。
BASELINE_FORMAL_SCORING_PLAN_RUN_ROOT = "/content/TSTW_runtime/runs/baseline_comparison_formal_scoring_plan"
BASELINE_SCORING_SHARD_COUNT = 1  # 外层 shard 总数。
BASELINE_SCORING_SHARD_INDEX = 0  # 当前运行的 shard 索引。
BASELINE_SCORING_BASELINE_NAMES = []  # scoring plan 阶段的 baseline 过滤器。空列表表示为 external_videoseal、external_rivagan、external_hidden_framewise 全部生成 work items。
RUN_BASELINE_FORMAL_SCORING_SMALL_VALIDATION = True  # 是否运行小规模 formal scoring execution。
BASELINE_FORMAL_SCORING_EXECUTION_RUN_ROOT = "/content/TSTW_runtime/runs/baseline_comparison_formal_scoring_execution"
BASELINE_FORMAL_SCORING_EXECUTION_BASELINE_NAMES = ["external_videoseal"]  # execution 阶段实际运行的 baseline 列表。
BASELINE_FORMAL_SCORING_EXECUTION_MAX_WORK_ITEMS = 2  # execution 最多执行的 work item 数。
BASELINE_FORMAL_SCORING_WORKER_COUNT = 1  # 三层策略第3层: 当前 shard 内并发 worker 数。
BASELINE_FORMAL_SCORING_BATCH_SIZE = 1  # 三层策略第3层: 每个 worker 一次领取的 work item 数。
BASELINE_GPU_PROFILE_INTERVAL_SECONDS = 0.5


In [ ]:
from pathlib import Path
import os
import subprocess
import sys


def run_command(command, cwd=None):
    """运行命令并在失败时显示 stdout 和 stderr。"""
    print("$", " ".join(command), flush=True)
    completed = subprocess.run(
        command,
        cwd=cwd,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )
    if completed.stdout:
        print(completed.stdout, flush=True)
    if completed.stderr:
        print(completed.stderr, file=sys.stderr, flush=True)
    if completed.returncode != 0:
        raise RuntimeError(f"命令失败: {' '.join(command)}")


def run_python_script(script, *args):
    """使用当前 Python 解释器运行仓库脚本。"""
    run_command([sys.executable, script, *args], cwd=REPO_DIR)


## 1. 挂载 Google Drive

该步骤只负责挂载 Drive。结果目录不会在这里提前创建, 只有 smoke 成功后才会复制完整结果包。

In [ ]:
try:
    from google.colab import drive
    drive.mount(DRIVE_MOUNT)
except Exception as exc:
    print(f"非 Colab 环境或 Drive 挂载失败: {exc}")


## 2. 获取项目仓库

如果 `REPO_URL` 为空, Notebook 假设 `/content/TSTW-VW` 已经存在。否则会从远程仓库冷启动 clone。


In [ ]:
repo_path = Path(REPO_DIR)
if REPO_URL and not repo_path.exists():
    run_command(["git", "clone", REPO_URL, REPO_DIR])
elif not repo_path.exists():
    raise FileNotFoundError("REPO_DIR 不存在, 请设置 REPO_URL 或先上传/克隆仓库。")

os.chdir(REPO_DIR)
if GIT_REF:
    run_command(["git", "fetch", "--all", "--tags"], cwd=REPO_DIR)
    run_command(["git", "checkout", GIT_REF], cwd=REPO_DIR)

print(f"当前仓库目录: {Path.cwd()}")
run_command(["git", "branch", "--show-current"], cwd=REPO_DIR)
run_command(["git", "rev-parse", "--short", "HEAD"], cwd=REPO_DIR)


## 2.1 检查并解压阶段二 real-video VAE 正式结果包

该步骤读取已经落盘到 Google Drive 的阶段二正式 family 结果目录, 检查 LPIPS、CLIP similarity、records、thresholds、tables 和 manifest 是否齐全。检查通过后, 将 zip 兼容包解压到 Colab 会话本地目录, 供后续正式 baseline comparison runner 读取。该步骤不会重跑 real-video VAE。


In [ ]:
stage_two_args = [
    "--package-root", REAL_VIDEO_VAE_PACKAGE_ROOT,
    "--summary-path", REAL_VIDEO_VAE_INPUT_CHECK_PATH,
]
if EXTRACT_REAL_VIDEO_VAE_PACKAGE:
    stage_two_args.extend(["--extract", "--extract-root", LOCAL_REAL_VIDEO_VAE_EXTRACT_ROOT])
run_python_script("scripts/prepare_baselines/check_real_video_vae_package_for_baseline.py", *stage_two_args)


## 3. 拉取外部 baseline 源码

`external_baselines/` 被 `.gitignore` 排除是合理的: 它属于第三方上游源码缓存, 不能作为本项目提交内容。每次 Colab 冷启动都通过下面的脚本按 manifest 中固定的 commit 重新拉取。

In [ ]:
run_python_script("scripts/prepare_baselines/fetch_external_baselines.py", "--print-plan")


## 3.1 安装 VideoSeal smoke 依赖

该步骤只安装 `external_videoseal` 真实 smoke 所需依赖。`external_baselines/` 仍然只是运行时缓存目录, 不会进入 git 提交。


In [ ]:
if RUN_VIDEOSEAL_REAL_SMOKE:
    run_command([
        sys.executable, "-m", "pip", "install",
        "-r", "external_baselines/external_videoseal/upstream/requirements.txt"
    ], cwd=REPO_DIR)


## 3.2 安装 RivaGAN smoke 依赖

该步骤只安装 `external_rivagan` 真实 smoke 所需的最小额外依赖。RivaGAN 上游代码硬编码 CUDA 调用, 因此该 smoke 需要 GPU runtime。


In [ ]:
if RUN_RIVAGAN_REAL_SMOKE:
    run_command([sys.executable, "-m", "pip", "install", "torch-dct==0.1.5"], cwd=REPO_DIR)


## 3.3 安装 HiDDeN framewise smoke 依赖

该步骤只安装 `external_hidden_framewise` 逐帧真实 smoke 所需的最小额外依赖。


In [ ]:
if RUN_HIDDEN_FRAMEWISE_REAL_SMOKE:
    run_command([sys.executable, "-m", "pip", "install", "opencv-python-headless"], cwd=REPO_DIR)


## 4. Source probe

该步骤检查上游源码树是否包含预期入口线索。它不下载权重, 也不运行真实模型。

In [ ]:
run_python_script("scripts/prepare_baselines/probe_external_baseline_sources.py")


## 5. Colab preflight

该步骤会显示当前真实 baseline 仍然缺少哪些条件, 包括权重 digest 和真实 adapter smoke。

In [ ]:
run_python_script("scripts/prepare_baselines/check_baseline_colab_preflight.py")


## 6. 运行 baseline smoke 并成功后落盘到 Drive

该命令先写入 `/content/TSTW_runtime/runs/`, 只有 smoke 成功且本地结果完整后, 才复制到 `/content/drive/MyDrive/TSTW/results/baseline_comparison_gate/<RUN_ID>/`。

In [ ]:
run_python_script(
    "scripts/prepare_baselines/run_baseline_comparison_smoke.py",
    "--run-root", SESSION_RUN_ROOT,
    "--result-root", DRIVE_RESULT_ROOT,
)


## 8. external_videoseal 真实可运行性 smoke

该步骤会真实加载 VideoSeal、下载权重、计算权重 digest、执行单视频 embed / detect, 并额外执行 H.264 CRF 28 后的 detect。该结果仍然只是单 baseline smoke, 不能替代正式 fixed-FPR baseline comparison。


In [ ]:
if RUN_VIDEOSEAL_REAL_SMOKE:
    videoseal_args = [
        "--run-root", "/content/TSTW_runtime/runs/external_videoseal_real_smoke",
        "--result-root", DRIVE_RESULT_ROOT,
        "--config-dir", BASELINE_CONFIG_DIR,
        "--external-root", EXTERNAL_BASELINES_ROOT,
        "--gpu-profile-interval-seconds", str(BASELINE_GPU_PROFILE_INTERVAL_SECONDS),
    ]
    if VIDEOSEAL_INPUT_VIDEO_PATH:
        videoseal_args.extend(["--input-video-path", VIDEOSEAL_INPUT_VIDEO_PATH])
    run_python_script("scripts/prepare_baselines/run_videoseal_real_smoke.py", *videoseal_args)
else:
    print("已跳过 external_videoseal 真实 smoke。")


## 9. external_rivagan 真实可运行性 smoke

该步骤会真实加载 RivaGAN、下载 32-bit 权重、计算权重 digest、执行单视频 encode / decode, 并额外执行 H.264 CRF 28 后的 decode。该结果仍然只是单 baseline smoke, 不能替代正式 fixed-FPR baseline comparison。


In [ ]:
if RUN_RIVAGAN_REAL_SMOKE:
    rivagan_args = [
        "--run-root", "/content/TSTW_runtime/runs/external_rivagan_real_smoke",
        "--result-root", DRIVE_RESULT_ROOT,
        "--config-dir", BASELINE_CONFIG_DIR,
        "--external-root", EXTERNAL_BASELINES_ROOT,
        "--gpu-profile-interval-seconds", str(BASELINE_GPU_PROFILE_INTERVAL_SECONDS),
    ]
    if RIVAGAN_INPUT_VIDEO_PATH:
        rivagan_args.extend(["--input-video-path", RIVAGAN_INPUT_VIDEO_PATH])
    run_python_script("scripts/prepare_baselines/run_rivagan_real_smoke.py", *rivagan_args)
else:
    print("已跳过 external_rivagan 真实 smoke。")


## 10. external_hidden_framewise 真实可运行性 smoke

该步骤会真实加载 HiDDeN 图像水印模型, 使用上游 combined-noise checkpoint, 对视频帧逐帧执行 encode / decode, 并额外执行 H.264 CRF 28 后的 decode。该结果仍然只是图像水印逐帧迁移 baseline smoke, 不能替代正式 fixed-FPR baseline comparison。


In [ ]:
if RUN_HIDDEN_FRAMEWISE_REAL_SMOKE:
    hidden_args = [
        "--run-root", "/content/TSTW_runtime/runs/external_hidden_framewise_real_smoke",
        "--result-root", DRIVE_RESULT_ROOT,
        "--config-dir", BASELINE_CONFIG_DIR,
        "--external-root", EXTERNAL_BASELINES_ROOT,
        "--gpu-profile-interval-seconds", str(BASELINE_GPU_PROFILE_INTERVAL_SECONDS),
    ]
    if HIDDEN_FRAMEWISE_INPUT_VIDEO_PATH:
        hidden_args.extend(["--input-video-path", HIDDEN_FRAMEWISE_INPUT_VIDEO_PATH])
    run_python_script("scripts/prepare_baselines/run_hidden_framewise_real_smoke.py", *hidden_args)
else:
    print("已跳过 external_hidden_framewise 真实 smoke。")


## 11. 汇总三个真实 smoke 结果包

该步骤读取 Drive 中最新的 `external_videoseal`、`external_rivagan` 和 `external_hidden_framewise` 真实 smoke 结果包, 生成 JSON、CSV 和 Markdown 摘要。该摘要只用于进入正式 baseline comparison 前的工程检查, 不支持论文 claim。


In [ ]:
if RUN_BASELINE_REAL_SMOKE_SUMMARY:
    run_python_script(
        "scripts/prepare_baselines/summarize_baseline_real_smoke.py",
        "--result-root", DRIVE_RESULT_ROOT,
        "--output-dir", BASELINE_REAL_SMOKE_SUMMARY_DIR,
    )
else:
    print("已跳过 baseline real smoke 汇总。")


## 12. 准备正式 baseline comparison runner 输入契约

该步骤读取已经解压的阶段二 records 和真实 smoke 汇总, 冻结正式 baseline comparison runner 的输入宇宙, 包括 split、sample role、attack name、内部方法变体和 target FPR。它仍然不生成论文主表, 只生成正式 runner 的输入契约。


In [ ]:
real_smoke_summary_json = f"{BASELINE_REAL_SMOKE_SUMMARY_DIR}/baseline_real_smoke_summary.json"
formal_input_short_commit = subprocess.check_output(
    ["git", "rev-parse", "--short", "HEAD"],
    cwd=REPO_DIR,
    text=True,
).strip()
run_python_script(
    "scripts/prepare_baselines/prepare_baseline_comparison_formal_inputs.py",
    "--stage-two-package-root", EXTRACTED_REAL_VIDEO_VAE_PACKAGE_ROOT,
    "--real-smoke-summary", real_smoke_summary_json,
    "--run-root", BASELINE_FORMAL_RUN_ROOT,
    "--result-root", DRIVE_RESULT_ROOT,
    "--short-commit", formal_input_short_commit,
)


## 13. 生成正式 baseline comparison scoring work-item 计划

该步骤读取 formal input contract 和阶段二 records, 生成外部 baseline 的正式 scoring work items, 并在成功后复制到 Drive。该步骤仍不伪造分数, 也不生成 TPR/FPR 表。后续执行层必须基于这些 work items 完成 embed、attack、detect、calibration 和 test 冻结。


In [ ]:
if RUN_BASELINE_FORMAL_SCORING_PLAN:
    formal_input_contract_path = f"{BASELINE_FORMAL_RUN_ROOT}/configs/baseline_comparison_formal_input_contract.json"
    scoring_args = [
        "--run-root", BASELINE_FORMAL_SCORING_PLAN_RUN_ROOT,
        "--stage-two-package-root", EXTRACTED_REAL_VIDEO_VAE_PACKAGE_ROOT,
        "--formal-input-contract", formal_input_contract_path,
        "--config-dir", BASELINE_CONFIG_DIR,
        "--external-root", EXTERNAL_BASELINES_ROOT,
        "--shard-count", str(BASELINE_SCORING_SHARD_COUNT),
        "--shard-index", str(BASELINE_SCORING_SHARD_INDEX),
        "--result-root", DRIVE_RESULT_ROOT,
        "--short-commit", formal_input_short_commit,
    ]
    for baseline_name in BASELINE_SCORING_BASELINE_NAMES:
        scoring_args.extend(["--baseline-name", baseline_name])
    run_python_script("scripts/prepare_baselines/run_baseline_comparison_formal_scoring.py", *scoring_args)
else:
    print("已跳过 baseline comparison formal scoring work-item 计划。")


In [ ]:
if RUN_BASELINE_FORMAL_SCORING_SMALL_VALIDATION:
    formal_input_contract_path = f"{BASELINE_FORMAL_RUN_ROOT}/configs/baseline_comparison_formal_input_contract.json"
    execution_args = [
        "--execute",
        "--run-root", BASELINE_FORMAL_SCORING_EXECUTION_RUN_ROOT,
        "--stage-two-package-root", EXTRACTED_REAL_VIDEO_VAE_PACKAGE_ROOT,
        "--formal-input-contract", formal_input_contract_path,
        "--config-dir", BASELINE_CONFIG_DIR,
        "--external-root", EXTERNAL_BASELINES_ROOT,
        "--shard-count", str(BASELINE_SCORING_SHARD_COUNT),
        "--shard-index", str(BASELINE_SCORING_SHARD_INDEX),
        "--max-work-items", str(BASELINE_FORMAL_SCORING_EXECUTION_MAX_WORK_ITEMS),
        "--worker-count", str(BASELINE_FORMAL_SCORING_WORKER_COUNT),
        "--batch-size", str(BASELINE_FORMAL_SCORING_BATCH_SIZE),
        "--result-root", DRIVE_RESULT_ROOT,
        "--short-commit", formal_input_short_commit,
    ]
    for baseline_name in BASELINE_FORMAL_SCORING_EXECUTION_BASELINE_NAMES:
        execution_args.extend(["--baseline-name", baseline_name])
    run_python_script("scripts/prepare_baselines/run_baseline_comparison_formal_scoring.py", *execution_args)
else:
    print("已跳过 baseline comparison formal scoring 小规模执行验证。")


## 14. 当前结论

如果上面的所有单元执行成功, 说明阶段二 real-video VAE 正式结果包可复用, 且阶段三三个外部 baseline 的 Colab 冷启动真实 smoke 链路已经具备工程可运行性。

- `external_videoseal` 和 `external_rivagan` 若在汇总中为 `real_smoke_passed`, 只能说明单视频 clean/H.264 smoke 可运行。
- `external_hidden_framewise` 若在汇总中为 `real_smoke_executed_negative`, 表示该图像水印逐帧迁移 baseline 可运行但低于 smoke 阈值, 应进入 limitation, 不能被写成原生视频水印成功 baseline。
- 下一步仍需实现并运行正式 fixed-FPR baseline comparison 评分 runner: 从本 Notebook 解压出的阶段二包读取同一 split、同一 attack matrix、calibration split 阈值依据、test split records、表格和 claim audit 所需输入。


## 9. 可选: 聚合全量 score records

该步骤仅在全量 shard score records 收齐后开启。


In [ ]:
if RUN_BASELINE_SCORE_RECORDS_AGGREGATION:
    if not BASELINE_SCORE_AGGREGATION_RECORD_PATHS:
        raise ValueError("BASELINE_SCORE_AGGREGATION_RECORD_PATHS 不能为空。请先填入全量 shard 的 score records 路径。")
    aggregation_args = [
        "--run-root", BASELINE_SCORE_AGGREGATION_RUN_ROOT,
        "--result-root", DRIVE_RESULT_ROOT,
        "--short-commit", formal_input_short_commit,
    ]
    for record_path in BASELINE_SCORE_AGGREGATION_RECORD_PATHS:
        aggregation_args.extend(["--record-path", record_path])
    run_python_script("scripts/prepare_baselines/aggregate_baseline_score_records.py", *aggregation_args)
else:
    print("已跳过 baseline score records 聚合。全量 shard 收齐后再开启。")
